# Training Meditation vs Thinking Classifier

This notebook demonstrates the complete pipeline for training a neural network classifier to distinguish between meditation and thinking tasks based on HRV features extracted from ECG signals.

**Tasks:**
- meditate1 (Label 1) vs think1 (Label 0)
- 98 total subjects, 4 tasks per subject

## 1. Setup and Imports

In [ ]:
import numpy as np
import os
from pathlib import Path
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Load environment variables
env_path = Path('.').resolve() / '.env'
load_dotenv(env_path)

# Import EEG processing modules
from eeg_processor import (
    Config, DataLoader, Preprocessor, HRVFeatureExtractor, HRVClassifier
)

print(f"BIDS Root: {os.getenv('BIDS_ROOT')}")

## 2. Configuration

In [ ]:
# Initialize configuration
config = Config(
    bids_root=os.getenv('BIDS_ROOT'),
    random_seed=42,
    ica_n_components=20,
    ica_method='fastica',
    filter_low_hz=1.0,
    filter_high_hz=40.0,
    psd_fmin=0.5,
    psd_fmax=50.0
)

# Task configuration
MEDITATE_TASK = 'meditate1'  # Label 1
THINK_TASK = 'think1'         # Label 0
MAX_SUBJECTS = 10            # Number of subjects to load

print(f"Configuration:")
print(f"  BIDS Root: {config.bids_root}")
print(f"  Tasks: {MEDITATE_TASK} (label 1) vs {THINK_TASK} (label 0)")
print(f"  Max subjects: {MAX_SUBJECTS}")

## 3. Load Multi-Subject Data

In [ ]:
# Initialize data loader
loader = DataLoader(config)

# Explore BIDS structure
print("\nExploring BIDS structure...")
bids_info = loader.explore_bids_structure()

# Load multiple subjects
print(f"\nLoading up to {MAX_SUBJECTS} subjects...")
loaded_data, failed = loader.load_multiple_subjects(
    session=None,
    task=None,
    max_subjects=MAX_SUBJECTS
)

print(f"\nLoaded: {len(loaded_data)} subjects")
print(f"Failed: {len(failed)} subjects")

## 4. Feature Extraction

Extract HRV features from ECG signals for each task.

In [ ]:
def extract_hrv_features_for_task(raw, task_name: str, ecg_channel: str = 'EXG7', sampling_rate: int = 500):
    """Extract HRV features for a specific task."""
    try:
        ecg_idx = raw.ch_names.index(ecg_channel)
        ecg_signal = raw.get_data(picks=ecg_idx)[0]
        
        extractor = HRVFeatureExtractor(sampling_rate=sampling_rate)
        features, _ = extractor.extract_with_interval(ecg_signal)
        
        return features
    except Exception as e:
        print(f"  [WARNING] Could not extract HRV for {task_name}: {e}")
        return None

print("Extracting HRV features for all subjects...\n")

all_features = []
all_labels = []
feature_names = ['rr_mean', 'sdnn', 'rmssd', 'pnn50', 'lf', 'hf', 'lf_hf', 'vlf']

for subject_id, raw in loaded_data.items():
    print(f"Subject {subject_id}:")
    
    # Set up electrode montage and channel types
    loader.raw = raw
    loader.setup_montage(montage_name='biosemi64', on_missing='ignore')
    
    channel_type_mapping = {
        'EXG1': 'eog', 'EXG2': 'eog', 'EXG3': 'eog', 'EXG4': 'eog',
        'EXG5': 'misc', 'EXG6': 'misc', 'EXG7': 'ecg'
    }
    loader.set_channel_types(channel_type_mapping)
    
    # Extract HRV for meditate1 task (label 1)
    meditate_features = extract_hrv_features_for_task(raw, MEDITATE_TASK)
    if meditate_features:
        print(f"  ✓ {MEDITATE_TASK}: extracted")
        all_features.append(list(meditate_features.values()))
        all_labels.append(1)  # Label 1 for meditate
    
    # Extract HRV for think1 task (label 0)
    think_features = extract_hrv_features_for_task(raw, THINK_TASK)
    if think_features:
        print(f"  ✓ {THINK_TASK}: extracted")
        all_features.append(list(think_features.values()))
        all_labels.append(0)  # Label 0 for think

X = np.array(all_features)
y = np.array(all_labels)

print(f"\n[OK] Extracted features for {len(X)} samples")
print(f"  - Shape: {X.shape}")
print(f"  - Classes: {np.bincount(y)}")
print(f"  - Think (0): {np.sum(y == 0)}, Meditate (1): {np.sum(y == 1)}")

## 5. Train/Val/Test Split

In [ ]:
# First split: 80% train+val, 20% test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Second split: 75% train, 25% val (of the 80% temp)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

print("Data split:")
print(f"  Train: {X_train.shape[0]} samples")
print(f"  Val:   {X_val.shape[0]} samples")
print(f"  Test:  {X_test.shape[0]} samples")
print(f"\n  Train - Think: {np.sum(y_train == 0)}, Meditate: {np.sum(y_train == 1)}")
print(f"  Val   - Think: {np.sum(y_val == 0)}, Meditate: {np.sum(y_val == 1)}")
print(f"  Test  - Think: {np.sum(y_test == 0)}, Meditate: {np.sum(y_test == 1)}")

## 6. Initialize and Train Classifier

In [ ]:
# Initialize classifier
classifier = HRVClassifier(
    input_size=X_train.shape[1],
    hidden_sizes=[64, 32],
    learning_rate=0.001
)

print(f"Classifier initialized:")
print(f"  Input size: {X_train.shape[1]} features")
print(f"  Hidden sizes: [64, 32]")
print(f"  Device: {classifier.device}")
print(f"\nStarting training...\n")

# Train the model
history = classifier.train(
    X_train, y_train,
    X_val, y_val,
    epochs=100,
    batch_size=16,
    feature_names=feature_names
)

## 7. Training History Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)

# Accuracy plot
axes[1].plot(history['train_acc'], label='Train Accuracy', linewidth=2)
axes[1].plot(history['val_acc'], label='Val Accuracy', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final metrics:")
print(f"  Train Loss: {history['train_loss'][-1]:.4f}, Train Acc: {history['train_acc'][-1]:.4f}")
print(f"  Val Loss:   {history['val_loss'][-1]:.4f}, Val Acc:   {history['val_acc'][-1]:.4f}")

## 8. Evaluate on Test Set

In [ ]:
# Make predictions on test set
test_predictions, test_probs = classifier.predict_with_proba(X_test)

# Calculate metrics
accuracy = accuracy_score(y_test, test_predictions)
precision = precision_score(y_test, test_predictions)
recall = recall_score(y_test, test_predictions)
f1 = f1_score(y_test, test_predictions)

print("\nTest Set Performance:")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1-Score:  {f1:.4f}")

## 9. Inference Examples with Probabilities

In [ ]:
print("\nSample predictions from test set:\n")

for i in range(min(10, len(X_test))):
    pred_class = test_predictions[i]
    pred_prob = test_probs[i]
    true_class = y_test[i]
    
    class_name = "Think" if pred_class == 0 else "Meditate"
    true_name = "Think" if true_class == 0 else "Meditate"
    correct = "✓" if pred_class == true_class else "✗"
    
    print(f"Sample {i+1}: {correct}")
    print(f"  True:        {true_name} ({true_class})")
    print(f"  Predicted:   {class_name} ({pred_class})")
    print(f"  Think prob:  {pred_prob[0]:.4f}")
    print(f"  Meditate prob: {pred_prob[1]:.4f}")
    print(f"  Confidence:  {max(pred_prob):.4f}\n")

## 10. Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, test_predictions)

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Think (0)', 'Meditate (1)'])
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title('Confusion Matrix - Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nConfusion Matrix:")
print(f"  Think -> Think:       {cm[0, 0]}")
print(f"  Think -> Meditate:    {cm[0, 1]}")
print(f"  Meditate -> Think:    {cm[1, 0]}")
print(f"  Meditate -> Meditate: {cm[1, 1]}")

## 11. Save Model

In [ ]:
# Save the trained model
model_dir = Path('.').resolve() / 'models' / 'meditation_classifier'
classifier.save(str(model_dir))

print(f"[OK] Model saved to: {model_dir}")
print(f"\nSaved files:")
print(f"  - model.pt")
print(f"  - scaler_mean.npy")
print(f"  - scaler_scale.npy")
print(f"  - metadata.json")

## 12. Summary

In [ ]:
print("\n" + "="*70)
print("TRAINING COMPLETE")
print("="*70)

print(f"\nDataset:")
print(f"  - Total samples: {len(X)}")
print(f"  - Training: {len(X_train)} samples")
print(f"  - Validation: {len(X_val)} samples")
print(f"  - Test: {len(X_test)} samples")

print(f"\nModel Performance (Test Set):")
print(f"  - Accuracy:  {accuracy:.4f}")
print(f"  - Precision: {precision:.4f}")
print(f"  - Recall:    {recall:.4f}")
print(f"  - F1-Score:  {f1:.4f}")

print(f"\nModel saved to: {model_dir}")
print(f"\nNext: Use inference.ipynb to make predictions on new data")